# Using LLMs as Agents

Our ultimate goal is to develop artificial intelligence systems that autonomously interact with its environment. We will call these **AI agents** or **agentic systems**. As of the time of writing, our best approach is to use a large language model (LLM) as the core computational engine for decision-making and planning of such agents (see papers section below). The rationale for this[^novelty] is that a lot of modern information infrastructure that humans have built are text-based (e.g. knowledge bases, documents, SQL) or can be reasoned with using text[^multimodal], and that current models are powerful enough to effectively plan and reason in natural language.

Agentic systems are architected such that the LLM itself dynamically reasons about and controls the sequence of actions. The model selects tools, evaluates outcomes, and iterates until a task is complete, making the path **autonomous**, **non-deterministic** and **goal-oriented**. We will discuss various patterns that takes advantage of this flexibility as well as create safeguards around it.[^tradeoff] This contrasts with more traditional uses of LLMs as components in a workflow setup (e.g. document translation + summarization pipeline).


[^multimodal]: Multi-modal LLMs extend the capabilities of agents to other modalities such as audio and images. Hence, the core LLM system not only reasons about text but also about the contents of images, audio, and video.

[^novelty]: The idea of autonomous agents is not exactly new; the novelty lies in that LLM-powered agents does this using an [informal]{.underline} and [inexact language]{.underline} (e.g. the [English language](https://x.com/karpathy/status/1617979122625712128)), but in a programmable, highly performant, and stochastic manner (creative?) &mdash; very much like humans.

[^tradeoff]: A key engineering trade-off is that this autonomy typically [increases latency]{.underline} and [cost]{.underline} compared to a single LLM call (e.g. in a fixed workflow), making architectural choice a critical design decision. For example, a program should always have an **upper bound** on the total number of tokens processed within a fixed time period.

<!-- ![**AI Agent building block.** Core LLM as engine for reasoning, planning, and tool calling. Agents follow goals and interact with their environment (external systems, such as other agents). This is not exactly new, the novelty lies in that AI agents does this using an [informal]{.underline} and [inexact language]{.underline} (e.g. the [English language](https://x.com/karpathy/status/1617979122625712128)), but in a highly performant and stochastic manner &mdash; very much like humans.](./img/augmented-llm.png){#fig-augmented-llm} -->

## Motivation

LLMs have two significant limitations: they are [stateless]{.mark} and prone to [hallucination]{.mark}. LLMs are fundamentally constrained by the **knowledge cutoff** from their [pre-training]{.underline} and struggle with [non-textual data]{.underline} (which is expected). Furthermore, LLMs suffer performance gaps between different task categories that make model selection difficult. 


### Hallucination

The core of the **hallucination problem** is that the pretraining data is larger than any model, and that we generally want minimally small models for efficiency, forcing LLMs to learn abstract representations. 
Internally, the sampling distributions becomes less definitive when the model encounters topics with scarce contextual or inconsistent training data[^quality]. This is worsened by knowledge cutoff, which renders information about recent events outdated or rare. When models hallucinate, the generated text still looks grammatically correct but may be stereotypical, follow frequent training-set patterns, or generate semantically close but incorrect concepts. This is concerning when non-experts use the system and can erode trust.

[^quality]: In general, an inabundance of training data coupled with poor quality data, bias, misinformation, and errors can cause models to hallucinate. LLMs do not have access to an external source of truth to validate the accuracy of their responses. They rely solely on their training data and the context provided in the input. If the model encounters ambiguous or poorly defined context, it may rely on its training data and generate content that is factually incorrect or hallucinated.

### Performance gaps

Another issue that LLMs have is performance gaps between different task categories (e.g. [GPQA](https://arxiv.org/abs/2311.12022), [MMLU](https://en.wikipedia.org/wiki/MMLU), [HumanEval](https://github.com/openai/human-eval), [MATH](https://huggingface.co/datasets/nlile/hendrycks-MATH-benchmark)). See @fig-llm-stats from [llm-stats](https://llm-stats.com/benchmarks). This adds  difficulty in choosing a model for a given task. The hope is that by equipping the LLMs with tools, the skill gap between same-sized models becomes insignificant. One approach is to dynamically change models or **triaging**[^humans] by using an initial model for categorizing the problem, then sending it to the appropriate specialized LLM.

[^humans]: A lot of agentic design pattern is inspired by how humans organize to solve tasks. For example, the triaging pattern is used in a hospital setting to deal with incoming patients with unknown afflictions while having doctors with deep specializations.

![**Model performance on various tasks.** Source: [llm-stats](https://llm-stats.com/benchmarks)](img/llm-stats.png){#fig-llm-stats}

### Prompting

Finally **prompt sizes** are limited finite resources. Llama 4 has an industry-leading (as of writing) context window of 10M, while Wikipedia consists of ~38B tokens. Moreover, as the context length grows, reasoning capabilities of LLM generally drops (@fig-needle_question_sim_arxiv). This motivates the practice of **context engineering** which refers to the set of strategies for curating and maintaining the optimal set of tokens (information) during LLM inference, including all the *other information* that may land there outside of the prompts (e.g. via retrieval / tool calls).

![**Decreasing performance for needle in a haystack setup.** The classic Needle in a Haystack task involves placing a random fact (the 'needle') in the middle of a long context window (the 'haystack'), then asking the model about that fact. Similarity refers to the similarity of the question to the needle. The degradation are observed for both long-context and usual models as well as across performance tiers.
](img/needle_question_sim_arxiv.png){#fig-needle_question_sim_arxiv}

![**Prompt engineering vs. Context engineering.** Context engineering involves systematically adding invoke-specific information for the LLM to consume. This typically includes system instructions (rules, behavior, e.g. tone), tool definitions, retrieved documents (RAG), conversation history or summaries of, memory, as well as structured data.
](img/context-engineering.png){#fig-context_engineering}


These limitations motivate the development of AI agents by equipping the core LLM services with external capabilities such as access to [APIs]{.underline}, [databases]{.underline} / [knowledge bases]{.underline}, as well as inducing [multi-step reasoning]{.underline} and [planning]{.underline} by creating elaborate **execution graphs** and **memory systems** or via post-training.

Finally, not all data are text-based or can be approached using verbal reasoning. Indeed, rudimentary forms of AI agents focus on solving these issues with LLMs first before even thinking about autonomy and action. Thus, LLMs are [augmented]{.underline}[^augment] with modules such as **goals** (to steer LLM outputs), **tools** (e.g. access to external APIs), and **memory** (statefulness). A more modest goal therefore of agent development is to design how LLMs interface with the modules including developing the modules themselves.

[^augment]: From a technical perspective, this augmentation is necessary for LLMs to reason about data outside of its training dataset (e.g. data outdated relative to the training cutoff), or nonverbal tasks that require specific, well-defined computation or algorithms, like search or specific computation such as root-finding or numerical integration. 

## Foundational papers

<!-- The rapid publication of **ReAct** [@ReAct2023], which is based on **Chain-of-Thought Prompting** [@cot], **Toolformer** [@toolformer], and **Generative Agents** [@genagents] within a few months of each other in early 2023 marks the period when LLMs transitioned from powerful stateless tools to the reasoning engines for autonomous, tool-using agents that we have today. These key papers laid the foundations for the capabilities of agentic systems. -->

### Chain-of-Thought [@cot]

CoT is a technique where a non-reasoning model is prompted by few-shot examples of step-by-step reasoning steps that lead to the final output (@fig-cot). This resulted in SOTA performance (at the time) on the [GSM8K benchmark](https://github.com/openai/grade-school-math) consisting of grade school math word problems, from 18% to 57% solve rate for the PaLM 540B [*just* by prompting]{.underline} (surpassing even a fine-tuned GPT3 at 33%) &mdash; the authors benchmarked the technique with other tasks such as robot instruction and commonsense reasoning, likewise getting significant improvement.

These results were interpreted as LLM's having **emergent abilities** that can be *unlocked* if just asked to do so explicitly. It also opened the paradigm that fine-tuning was not necessary to get SOTA performance. Finally, since CoT only involves a change in prompt design, it was easily applicable to every LLM use-case.

![**Figure 1 of** [@cot]. **Few-shot and Zero-shot CoT.** Chain-of-thought prompting enables LLMs to tackle complex arithmetic, commonsense, and symbolic reasoning tasks just by a [simple trick]{.underline}. Chain-of-thought reasoning processes are highlighted.](img/zero-cot.png){#fig-cot}

### ReAct [@ReAct2023]

CoT (the "reasoning" part) was further augmented by **acting** (e.g. action
plan generation) in this paper. This paper 
explored the use of LLMs to generate both reasoning traces and task-specific actions in an interleaved manner resulting in greater synergy: reasoning traces help the model to play around action plans (e.g. handle exceptions), while actions allow it to interface with external sources (knowledge bases, Wikipedia) thereby gathering additional information (@fig-react-fig1). 

Performing actions resulted in improved **groundedness** and **trustworthiness** for the system: authors report 0% of ReAct failures are hallucinations, compared to 56% for CoT, instead 47% failures for ReAct are due to wrong reasoning traces including repetitive steps. In the paper, ReAct shows strong generalization to new task instances with only 1-6 **few-show examples** (i.e. manually composed ReAct-format trajectories, no fine-tuning[^react-finetuning]), consistently outperforming only reasoning or only acting baselines across different domains. 

![**Figure 1 of** [@ReAct2023]. The combination of reasoning and acting allows the agent to solve the task vs. reason-only and act-only approaches. ReAct outperforms imitation and reinforcement learning methods on two interactive decision making benchmarks (ALFWorld and WebShop) by an absolute success rate of 34% and 10% respectively, while being prompted with only one or two in-context examples.](./img/react-paper.png){#fig-react-fig1}

[^react-finetuning]: The authors attempted to fine-tune a smaller model PaLM-8/62B via **distilling** ReAct-format reasoning traces from PaLM-540B. That is, instead of conditioning the model via prompts, the reasoning traces are baked into the generating process. The resulting trained model outperformed all 540B pure-prompting methods. The authors believe that finetuning with more human-written data may unleash the full potention of ReAct.

### Toolformer [@toolformer] 

The next paper focused on **tool calling**. At this point, LLMs have been shown to be proficient in few-shot prompting. But it still struggles with basic tasks like arithmetic or [counting r's in "strawberry"](https://techcrunch.com/2024/08/27/why-ai-cant-spell-strawberry/). Moreover, it's knowledge is frozen after training cutoff, and as such is outdated by the time it gets to the user. All these issues are solved if the model is able to use tools. 

In the Toolformer paper, it is shown how a language model can be **trained** for **API calling** without fundamentally changing the model's architecture or training objective, and without a specialized dataset. The model learns to decide *which* API to call, *when* to call it, *what* arguments to pass, and *how* to best incorporate the results into the token generation process. The goal is to insert API calls into text generation. This were done in the ff. steps:

1. **Self-supervised data generation.** The key innovation of the paper is that the API calls are generated without needing human labelers. The model itself to generate potential API calls from text. The language model is exposed to few-shot examples of API calls inserted in [helpful places]{.underline} (the *when*). From Appendix A.2 of the paper, we see the prompts used. Here's one for the calculator tool:

    ```
    Your task is to add calls to a Calculator API to a piece of text. The calls sho
    uld help you get information required to complete the text. You can call the AP
    I by writing "❬Calculator(expression)❭" where "expression" is the expression to
     be computed. Here are some examples of API calls:
    
    Input:  The number in the next term is 18 + 12 x 3 = 54.
    Output: The number in the next term is 18 + 12 x 3 = ❬Calculator(18 + 12 *3)❭ 5
    4.

    Input:  A total of 252 qualifying matches were played, and 723 goals were score
    d (an average of 2.87 per match). This is three times less than the 2169 goals 
    last year.
    Output: A total of 252 qualifying matches were played, and 723 goals were score
    d (an average of ❬Calculator(723 / 252)❭ 2.87 per match). This is twenty goals 
    more than the ❬Calculator(723 - 20)❭ 703 goals last year.
    
    Input: {x}
    Output:
    ```

    Then, for an input sequence $\mathbf{x}$ we iterate over positions $i$ in the text and save those where 
    the model conditioned on $[\Phi(\mathbf{x}), \mathbf{x}_{<i}]$ assigns a high probability of an API call $\boldsymbol{\langle}$ as 
    next token[^sampling_threshold]. Here $\Phi(\mathbf{x})$ is the accompanying tool call prompt (see above) &mdash; so that the model sees the entire input, giving it hints on what the expected
    API call output should be.
    Next, for each such position we get a number of API args $\mathbf{c}^1_i, \ldots, \mathbf{c}^m_i$ (one tool with different arguments) by generating text given $[\Phi(\mathbf{x}), \mathbf{x}_{<i}, \boldsymbol{\langle}]$ until the model outputs $\boldsymbol{\rangle}$ as EOS token (examples that do not generate this are discarded). Note that [no actual execution]{.underline} is performed so far. We only want a sample of positions along with API args for that position for each input $\mathbf{x}.$[^phase1]

    ![**Examples of API calls and responses.** The tools are designed to take natural language as input. The tool specification influence the design of few-shot prompts $\Phi.$ See paper for details (e.g. the MT tool serves [NLLB-600M](https://huggingface.co/facebook/nllb-200-distilled-600M) a multi-lingual machine translation model for 200 languages with [fastText](https://github.com/facebookresearch/fastText) to identify the source language).](./img/toolformer-apicalls.png)

    In the next phase, we perform the executions of the sampled API args $\mathbf{c}^1_i, \ldots, \mathbf{c}^m_i$. For $l = 1, \ldots, m$, the response needs to be a single text sequence $\mathbf{r}_i^l$ (this includes error responses). Finally, we filter out the sample API calls by evaluating to see if the tokens of $\mathbf{x}$ after $i$ is assigned a significantly lower loss (vs. some error threshold[^api_threshold] $\tau_f$) vs. the generation [without]{.underline} tool call[^actual]. 
    The API calls with their execution results are then interleaved in the text to get $\mathbf{x}^* = [\mathbf{x}_{<i}, \boldsymbol{\langle}\mathbf{c}_i \rightarrow \mathbf{r}_i \boldsymbol{\rangle}, \mathbf{x}_{\geq i} ].$
    Doing this process of candidate generation, execution, and filtering (@fig-toolformer) to every text in the original dataset yields the final, curated dataset used for fine-tuning. 
    Some examples in the fine-tuning dataset[^ft-format]:

    ```
    The name derives from “la tortuga”, the Spanish word for ❬MT(“tortuga”) → turtle❭ turtle.
    Out of 1400 participants, 400 (or ❬Calculator(400 / 1400) → 0.29❭ 29%) passed the test. 
    ```
    
2. **Standard fine-tuning & modified inference.** The language model is then trained on the augmented tool-use dataset using standard techniques. Note that the output of the API calls are already in the dataset so we don't have to perform any execution at this stage. However, during inference the engine is designed such that whenever the token `→` is detected, the decoding is paused to execute the API call and the generation continues with the result and the EOS `❭` added to the context. The authors used greedy decoding with `❬` selected if it's in the top-*k* of candidate tokens to encourage tool-calling, with limitations on the amount of tool calls per input. The model is expected to perform tool-calling in a **zero-shot** setup (i.e. no examples of tool-calling are needed in its prompt.)

![**Figure 2 of** [@toolformer]. A language modeling dataset is augmented with API calls. Instead of just predicting "the Steel City", an API call is created as an in-between target.](./img/toolformer.png){#fig-toolformer}

[^sampling_threshold]: The top-$k$ positions in the text $\mathbf{x}$ with next-token probability of an API call that exceeds $\tau_s = 0.05$ or 5%. For large datasets, $k=5$ and $m=5$ were used. For smaller, this is increased to $k=20$, $m=10$, and $\tau_s = 0.$

[^phase1]: This phase focuses on gathering candidate data that can potentially teach a model on 
*when* to call and API and *which* API arguments to call. Not so much how the actual return values of the 
call affects future generation.

[^api_threshold]: The size of the generated dataset decrease rapidly as $\tau_f$ increases (see Table 2 in the paper).

[^actual]: Suppose the API call happens at index $i$ in a sequence $\mathbf{x}$, the model is conditioned
on $[\mathbf{z}, \mathbf{x}_{<i}]$ and a weighted cross-entropy of the ground truth sequence $\mathbf{x}_{\geq i}$ is calculated, with the prefix $\mathbf{z}$ being either (1) FULL `❬f(args) → r❭` where $\mathbf{c}_i =$ `f(args)` and $\mathbf{r}_i =$ `r`, (2) MASKED RESULT `❬f(args) → ❭` and (3) NO API call (i.e. the usual generation). A sample API call is kept only if the weighted cross-entropy of $\mathbf{x}_{\geq i}$ (1) is the minimum compared to (2) and (3) by at least $\tau_f.$ Machine Translation (`MT`) tool example:
```
(1) [❬MT(“tortuga”) → turtle❭ The name derives from “la tortuga”, the Spanish word for] turtle.
(2) [❬MT(“tortuga”) → ❭ The name derives from “la tortuga”, the Spanish word for] turtle.
(3) [The name derives from “la tortuga”, the Spanish word for] turtle.
```
This intuitively means that the act of calling a particular API is not only useful, but its result is also useful. Here the API call $\mathbf{z}$ is added as a prefix in getting $p_\textsf{model}(\mathbf{x}_{\geq i} \mid [\mathbf{z}, \mathbf{x}_{<i}])$ is so that the complete sequence $[\mathbf{z}, \mathbf{x}_{<i}, \mathbf{x}_{\geq i}]$ remains in-distribution w.r.t. the pretraining data.
<br><br>
Finally, the cross-entropy weights $w_t \geq 0$ where $t = j - i$ decay for increasing $t.$ 
The authors used $w_t = \max(0, 1 - 0.2 \cdot t)$ which means only 6 next tokens contribute to the loss. 
These are normalized over the sequence, i.e. the actual weights
used are $\tilde{w}_t = w_t / \sum_{t^\prime} w_{t^\prime}.$
Using decaying weights ensures that API calls happen close to where
the information provided by the API is actually helpful for the model. 

[^ft-format]: This format is exactly how API calls during the generation process are performed. In the actual paper, `[`, `]`, and `->` are used so that everything works without modifying the existing vocabulary. 

The self-supervised data generation process relies on base model already possessing a rich, implicit knowledge of tools and their purposes from its pre-training (humans use tools intuitively, and tool use follows causal and physical rules, all of these reflect in recorded language data, so it's likely that LLMs also learn it from the statistics of the data). Assuming this is true, what the LLM then lacks is the 'grammar' for tool calls[^grammar_tool_call]. The fine-tuning phase solved this by forging a new, high-probability pathway into its output generation (i.e. using the tool call token $\text{❬}$ and forming proper arguments) in contexts where information has to be identified from external sources (@fig-toolformer-evals).

![**Performance of Toolformer in downstream tasks.** Toolformer was evaluated on a variety of downstream tasks. A zero-shot setup is used where no tool-calling examples were provided in the prompt. Greedy decoding was used with an API call triggered when $\text{❬}$ is found in the top-10 next token candidates. Toolformer with [tool-calling disabled]{.underline} was also evaluated to ensure that the base performance did not degrade (i.e. forcing $p(\text{❬}) = 0$). GPT-J + CC refers to [GPT-J](https://en.wikipedia.org/wiki/GPT-J) (6B) fine-tuned on a subset of [Common Crawl](https://en.wikipedia.org/wiki/Common_Crawl), and Toolformer is the same GPT-J fine-tuned on CC$^*$ (i.e. Common Crawl augmented with API calling discussed). Interestingly, Toolformer with API calling disabled performs better than GPT-J + CC &mdash; it is surmised that being exposed more API calls and their results during training improved its own reasoning abilities[^pure-lm-perf]. Effect of tool calling is also emphasized by beating GPT-3 (175B) a significantly larger model in these tasks.](./img/toolformer-evals.png){#fig-toolformer-evals}

[^grammar_tool_call]: Toolformer translated the tool-use problem into a pure text completion problem, and showed it was possible. Without having read the paper, I would think an RL-based approach would be required to successfully solve this. However, from the empirical analysis of scaling laws in the paper (**Figure 4**), it was shown that Toolformer ([GPT-2]{.underline} fine-tune) only beats Toolformer (disabled &mdash; i.e. forcing $p(\text{❬}) = 0$) past **775M parameters**, implying that tool-calling capabilities started to emerge once language ability was sufficiently high. The scaling laws also show that the gap between Toolformer and Toolformer (disabled) does not decrease past the parameter count where the model learns tool-calling.

[^pure-lm-perf]: How about performance on pure language modeling (e.g. "Write me a story")? **Table 8** of the paper shows base language modeling performance of the models in terms of perplexity. It is shown that Toolformer (disabled) (+ CC$^*$) did not degrade relative to GPT-J + CC which is the valid comparison. It did degrade in [WikiText](https://huggingface.co/datasets/Salesforce/wikitext) which is fine since GPT-J + CC also did, so this may be just due to distribution shift.

### Generative agents [@genagents]

Finally, the last paper *Generative Agents: Interactive Simulacra of Human Behavior* dives into the ability of agents to simulate believable human behavior. This highlights the ability of LLM-based agents to interact with their **environment** which includes other agents. Hence, providing an example of a working multi-agent architecture. The agents in this paper are characterized as being able to **remember**, **reflect**, and **plan** based on growing memory and casading social dynamics. In the paper, a sandbox environment are filled with 25 LLM-based agents that are shown to demonstrate believable human behavior (@fig-genagents). 

![**Figure 1 of** [@genagents]. Generative agents socialize, chill out, talk about politics & news, and perform routines. Moreover they have been observed to perform longer-term planning and coordinate with other agents in their plans.](./img/genagents.png){#fig-genagents}

**Generative agents.** Each agent is [seeded]{.underline} with a paragraph in natural language that depict their identity (including their occupation and relationship with other agents). Each `;`-delimited phrase is entered into the agent’s **initial memory** as memories at the start of the simulation (later we will discuss the memory stream in more detail). 

Generative agents operate in
an action loop where, at each time step, they perceive the world
around them and perform actions, or talk to other agents. 
Consider the agent John Lin. John Lin
interacts with their world in several key ways:

1. **Action.** At each time step, an agent outputs a natural language statement describing its current action ("John Lin is researching the local mayor election"). This includes interaction with their environment (e.g. "closet is being used to select clothes for the day"). Agents can also move between different locations in the map. 

2. **Observation.** Agents can record the state of their local environment in their memory ("refrigerator is idle"). This includes other agents: "Adam Smith is discussing the topic with the other participants". 

3. **Dialogue.** Agents converse as they interact with each other. Agents' dialogue are generated by conditioning 
on their memory of each other and the current summary status of the agent, as well as their observation of the 
other agent whom they want to engage in a conversation with:

    ```
    [Agent's Summary Description:]
    It is February 13, 2023, 4:56 pm. 
    John Lin's status: John is back home early from work. 

    [Observation:] 
    John saw Eddy taking a short walk
    around his workplace.

    [Summary of relevant context from John's memory:]
    Eddy Lin is John's Lin's son. Eddy Lin has been
    working on a music composition for his class. Eddy
    Lin likes to walk around the garden when he is
    thinking about or listening to music.
    John is asking Eddy about his music composition
    project. 

    What would he say to Eddy?
    ```

The continuation of this dialogue is generated using
the same mechanism until one of the two agents decides to end the
dialogue. Below (in the *reacting* discussion) it is discussed in more detail how inter-agent dialogues are triggered.

:::{.callout-tip}
## Emergent behavior
These interactions result in emergent behavior such as [information diffusion]{.underline} and [coordination]{.underline} (planning an event involving multiple agents helping each other out and arriving on time at a specific location).

:::


**Memory and retrieval.** Everything that an agent experiences is recorded and reasoned over as a natural language description, 
allowing the architecture to leverage a LLM. 
Generative agents take their current environment and past experiences as [input]{.underline} and generate behavior as output.
Underlying this behavior is a novel agent architecture that combines a large language model with mechanisms
for synthesizing and retrieving relevant information to condition
the language model’s output.

- **Memory stream.** This is a list of 
memory objects for each agent that contains a timestamp created, timestamp of last access (which defaults to created), 
and a natural language description. 
Observations, i.e. event directly perceived by agent (e.g. behavior of self, other, non-agent objects)
are stored in memory. Conversation between agents are summarized in the memory stream.

- **Retrieval function.** Because generative agents produce large streams of events and memories
that must be retained, a core challenge is ensuring that the most relevant pieces of the agent's memory are
retrieved and synthesized when needed. For a query $Q$, we retrieve memory items $m$ based on the score:
    $$\text{score}(m, Q) = \alpha_1 \cdot 0.995^{\Delta t_m} + \alpha_2 \cdot \text{I}(m) + \alpha_3 \cdot (\hat{\mathbf{v}}_Q \cdot \hat{\mathbf{v}}_m)$$
    where $\alpha_1 + \alpha_2 + \alpha_3 = 1.$
    Here $\Delta t_m$ is the time elapsed from the last access of $m$, so that the first term is a *recency* score that decays exponentially. 
    For example, events that occured on the same day are more likely to be retrieved, all things being equal, 
    compared to events that occured on the previous week. Next, the *importance* $\,\text{I}(m)$ is determined by the LLM itself
    by prompting:

    ```
    On the scale of 1 to 10, where 1 is purely mundane (e.g., brushing teeth, mak
    ing bed) and 10 is extremely poignant (e.g., a break up, college acceptance),
     rate the likely poignancy of the following piece of memory.
    
    Memory: buying groceries at The Willows Market and Pharmacy
    Rating: <fill in>
    ```

    Finally, we calculate relevance as the cosine similarity between the memory's 
    embedding vector $\hat{\mathbf{v}}_m$ and the query memory's embedding vector $\hat{\mathbf{v}}_Q$ using the LLM. Here unit vectors are 
    used, so we just dot these to get cosine similarity. The top-ranked memories that fit within
    the language model’s context window are included in the prompt upon which the retrieval function is used
    to fill with memory items.


![**Figure 6 of ** [@genagents]. The memory stream consists of a large number of observations that are relevant and irrelevant to the agent's current situation. Retrieval identifies a subset of these observations that should be passed to the language model to condition its
response to the situation.](./img/genagents-memory-retrieval.png)


**Reflection.** 
Generative agents, when equipped with only raw observational memory, struggle to generalize or make inferences.
Reflections are introduced as a second type of memory (e.g. the retrieval function also returns reflections). 
Reflections are higher-level, more abstract thoughts
generated by the agent. Because they are a type of memory, they
are included alongside other observations when retrieval occurs. In the paper, this triggers for an agent
when the sum of importance scores exceed 150. This is done as follows:

1. **Three points of reflection.** Query the 100 most recent memory. Prompt the language model: "Given only the information above, what are 3 most salient high-level questions we can answer about the subjects in the statements?". This method directly extracts specific lines of inquiry rather than producing a general summary.

2. **Reflection proper.** For one question $Q$ at a time, retrieve memories using the retrieval function $f(Q).$ Prompt the language model to extract insights:
    
    ```text
    Statements about Klaus Mueller
    1. Klaus Mueller is writing a research paper
    2. Klaus Mueller enjoys reading a book
    on gentrification
    3. Klaus Mueller is conversing with Ayesha Khan
    about exercising [...]

    What 5 high-level insights can you infer from the above statements? 
    (example format: insight (because of 1, 5, 3))
    ```

3. **Extend reflection tree.** Store these in the memory stream and cite the particular records that served as evidence for the insights. Since reflections are memories, agents can reflect on past reflections.
As a result, agents generate trees of reflections (@fig-reflections-tree): the [leaf nodes]{.underline} of
the tree represent the base observations, and the non-leaf nodes
represent [thoughts]{.underline} that become more abstract and higher-level the
higher up the tree they are. Hence, we can assign a **depth** value to each memory.

![**Figure 7 of ** [@genagents] A reflection tree for Klaus Mueller. The agent’s observations of the world, represented in the leaf nodes, are recursively synthesized to derive Klaus’s self-notion that he is highly dedicated to his research.](./img/reflections-tree.png){#fig-reflections-tree}


**Planning and reacting.** Agents need to plan over a longer time horizon to ensure that their sequence
of actions is coherent and believable. A **plan** includes a location, a starting time, and a duration.
Like reflections, plans are [stored in the memory stream]{.underline} and are included in the retrieval process. This
allows the agent to consider observations, reflections, and plans all together when deciding how to behave. 

First, at the start of each day, the language model is prompted with the agent's summary description (e.g. name, traits, and a summary of their recent experiences) and a summary of their previous day. A general plan for the day's agenda is obtained (7-8 items). These are then recursively decomposed to create finer-grained actions, first into hour-long chunks of actions, and then to 5-15 minute chunks (@fig-genagents-plan).

Agents may [change plans]{.underline} mid-stream if needed given observations from their environment. We prompt the language model with these observations to decide whether the agent should continue with their existing plan, or **react**. The prompt template is given by:
```text
[Agent's Summary Description:] 
{timestamp,agent_current_state}

[Observation:] 
{observation} 

[Summary of relevant context from John’s memory:] 
{context_summary}

Should John react to the observation, and if so, 
what would be an appropriate reaction?
``` 
Let `O` be the observer and `E` be the observed entity. The context is generated through two prompts that retrieve memories by summarizing the combined output of the two queries with the retrieval function: `"What is [O]'s relationship with [E]?"` and `"[E] is [status_of_E]"`.
Note that these include plans, since plans are also in the memory stream.
We then regenerate the agent's existing plan starting from the time when the reaction takes place. Finally, if the output action indicates an interaction between agents, we generate their dialogue.

![**Planned tasks of Yuriko Yamamoto.** Current status: Yuriko Yamamoto is working on a tax compliance project for a local business. She is also taking classes to stay up to date on new tax laws. Yuriko is also curious about who will be running for the local mayor election next month.](./img/genagents-plan.png){#fig-genagents-plan}

:::{.callout-note}
**Generative Agents** [@genagents] showed that LLMs can simulate believable behavior by having (1) a [seeded agent identity]{.underline}, and (2) a [core memory module]{.underline} that consist of observations, reflections, and current plans. Since memories are noisy and will quickly fill LLM context, having a [retrieval function]{.underline} is crucial given the query that applies to the current situation. 
This is augmented by having [reflections]{.underline} which are abstract observations that compress previous events and that were recollected based on recent events. 
See Appendix B of the paper where agents are interviewed (e.g. *Give an introduction of yourself*) which can 
be answered using the retrieval function.
Finally, allowing LLMs to interact results in [emergent phenomenon]{.underline} (i.e. occurences that not explicitly built into the system).

:::

## Code Experiments

The three foundational papers above each make a concrete empirical claim. We now run minimal experiments that reproduce these claims from scratch &mdash; no frameworks, no abstraction layers &mdash; to build direct intuition for what each technique actually does and why it matters.

### Experiment 1: Chain-of-Thought Effect

The CoT paper's central claim is that prompting alone &mdash; no fine-tuning, no change in weights &mdash; can unlock substantially better reasoning. We test this on 10 problems from the [GSM8K benchmark](https://github.com/openai/grade-school-math), comparing a **direct** prompt against a **chain-of-thought** prompt using the same model.

**Setup.** We load 10 GSM8K test problems. For each, we send two requests to the same model (`openai/gpt-4o-mini`) with different system prompts: one that demands only a number, and one that asks the model to think step by step before answering.

In [ ]:
import os
import re
from datasets import load_dataset
from dotenv import load_dotenv
from openai import AsyncOpenAI

load_dotenv()

client = AsyncOpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url=os.environ.get("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"),
)


async def chat(messages, model="openai/gpt-4o-mini", temperature=0, **kwargs):  # (1)
    response = await client.chat.completions.create(
        model=model, messages=messages, temperature=temperature, **kwargs,
    )
    return response.choices[0].message.content


dataset  = load_dataset("openai/gsm8k", "main", split="test[:10]")
problems = [
    {"question": row["question"], "answer": row["answer"].split("####")[1].strip()}
    for row in dataset
]

DIRECT_SYSTEM = "Answer with just a number. No explanation."
COT_SYSTEM    = "Think step by step, then end your response with 'Answer: <number>'."

def extract_number(text: str) -> str | None:
    """Return the last integer or decimal (commas stripped) found in text."""
    matches = re.findall(r"-?\d+(?:,\d{3})*(?:\.\d+)?", text)
    return matches[-1].replace(",", "") if matches else None

We run both prompt conditions for every problem and record whether the predicted answer matches the gold label:

In [ ]:
results = []

for p in problems:
    q, gold = p["question"], p["answer"]

    direct_reply = await chat([
        {"role": "system", "content": DIRECT_SYSTEM},
        {"role": "user",   "content": q + "\n\nAnswer:"},
    ])

    cot_reply = await chat([
        {"role": "system", "content": COT_SYSTEM},
        {"role": "user",   "content": q},
    ])

    results.append({
        "question":    q[:60] + "...",
        "gold":        gold,
        "direct_pred": extract_number(direct_reply),
        "cot_pred":    extract_number(cot_reply),
        "direct_ok":   extract_number(direct_reply) == extract_number(gold),
        "cot_ok":      extract_number(cot_reply) == extract_number(gold),
    })

**Results.** Displaying correctness per problem and aggregate accuracy:

In [ ]:
#| code-fold: true
import pandas as pd

df = pd.DataFrame(results)
df["Direct"] = df["direct_ok"].map({True: "✓", False: "✗"})
df["CoT"]    = df["cot_ok"].map({True: "✓", False: "✗"})

display(
    df[["question", "gold", "direct_pred", "Direct", "cot_pred", "CoT"]]
      .rename(columns={
          "question":    "Question",
          "gold":        "Gold",
          "direct_pred": "Direct Pred",
          "cot_pred":    "CoT Pred",
      })
)

n = len(results)
direct_acc = sum(r["direct_ok"] for r in results)
cot_acc    = sum(r["cot_ok"]    for r in results)
print(f"Direct accuracy : {direct_acc}/{n}")
print(f"CoT accuracy    : {cot_acc}/{n}")

The same model, the same weights, zero retraining &mdash; the only difference is [what we asked it to do]{.underline}. CoT unlocks a latent reasoning capability the model already possesses but does not express unless prompted. This is the empirical basis behind the "emergent abilities" framing in the paper.

### Experiment 2: ReAct Trace from Scratch

We implement the **ReAct loop** from first principles &mdash; no frameworks. The key insight is that ReAct is simply a conversation: the model outputs `Thought:` / `Action:` blocks, we parse and execute the action, inject the `Observation:`, and loop until the model outputs `Final Answer:`. The tool here is a live Wikipedia lookup via the REST API.

**Setup.** Defining the Wikipedia search tool and the ReAct system prompt that instructs the model to use the `Thought` / `Action` / `Final Answer` format:

In [ ]:
import requests

def search_wikipedia(query: str, chars: int = 400) -> str:
    """Return the first `chars` characters of a Wikipedia article summary."""
    url = "https://en.wikipedia.org/api/rest_v1/page/summary/" + requests.utils.quote(query)
    resp = requests.get(url, headers={"User-Agent": "ai-notebooks/1.0"}, timeout=10)
    if resp.status_code != 200:
        return "[No Wikipedia article found for that query.]"
    return resp.json().get("extract", "[No extract available.]")[:chars]

REACT_SYSTEM = """You are an assistant that answers questions by thinking step-by-step and using tools.

For each step, output EXACTLY in this format:
Thought: [your reasoning]
Action: search[\"your search query\"]

When you have the answer, output:
Final Answer: [your answer]

Only use the 'search' action. Be concise."""

Running the ReAct loop and printing each step as it executes. We use `meta-llama/llama-3.3-70b-instruct` via OpenRouter to show that the same client can target different providers:

In [ ]:
REACT_MODEL = "meta-llama/llama-3.3-70b-instruct"

TASK = "Who was the US President when the Berlin Wall fell, and what is their most famous quote?"

history = [
    {"role": "system", "content": REACT_SYSTEM},
    {"role": "user",   "content": TASK},
]

print(f"Task: {TASK}\n" + "-" * 60)

for step in range(5):  # <1>
    reply = await chat(history, model=REACT_MODEL)
    history.append({"role": "assistant", "content": reply})
    print(reply.strip())

    if "Final Answer:" in reply:  # <2>
        break

    match = re.search(r'search\[\"(.+?)\"\]', reply)
    if match:  # <3>
        query       = match.group(1)
        observation = search_wikipedia(query)
        obs_msg     = f"Observation: {observation}"
        print(obs_msg)
        history.append({"role": "user", "content": obs_msg})
    else:
        break

    print()

1. A hard cap at 5 steps bounds token usage; in practice the loop terminates earlier on `Final Answer:`.
2. Once the model commits to a final answer, we stop &mdash; no further tool calls are needed.
3. Injecting the `Observation:` into the history is what grounds the next `Thought:` in retrieved evidence rather than parametric memory. Groundedness is not a property of the model &mdash; it is [a property of the context the model is given]{.underline}.

:::{.callout-note}
The ReAct paper reports 0% hallucination failures for ReAct vs. 56% for CoT-only on knowledge-intensive tasks. The trace above illustrates why: every factual claim in the final answer is anchored to a concrete `Observation` from an external source. The model cannot hallucinate what it was just shown.

:::

### Experiment 3: Hallucination Without Grounding

Tool access is not primarily about capability &mdash; it is about **groundedness**. A model without tools must answer from parametric memory alone; when that memory is sparse, stale, or inconsistent, the model confabulates. We compare the same model answering 5 factual questions with and without Wikipedia access.

**Setup.** Five questions chosen to probe different failure modes: obscure geography, biology (common misconception), a trick question about a store still open, astronomical distance, and a baseline where the model almost certainly knows the correct answer:

In [ ]:
QUESTIONS = [
    {
        "q":       "What is the capital of the least populous country in Africa by land area?",
        "correct": "Moroni (Comoros)",
    },
    {
        "q":       "How many bones does a shark have?",
        "correct": "Zero — sharks have cartilage, not bone",
    },
    {
        "q":       "What year did the last Blockbuster store close?",
        "correct": "It has not closed — one store in Bend, Oregon remains open as of 2024",
    },
    {
        "q":       "What is the approximate distance from Earth to Proxima Centauri, in light-years?",
        "correct": "~4.24 light-years",
    },
    {
        "q":       "What programming language is the Linux kernel written in?",
        "correct": "C",
    },
]

**No-tool condition.** The model answers from parametric memory only &mdash; no retrieval, no tool calls:

In [ ]:
NO_TOOL_SYSTEM = "Answer the question as concisely as possible. Give only the factual answer."

no_tool_answers = []
for item in QUESTIONS:
    reply = await chat([
        {"role": "system", "content": NO_TOOL_SYSTEM},
        {"role": "user",   "content": item["q"]},
    ])
    no_tool_answers.append(reply)

**With-tool condition.** A minimal single-step dispatch loop: the model may optionally issue one `search[...]` call; we execute it against the Wikipedia API and return the observation, then ask for the final answer:

In [ ]:
WITH_TOOL_SYSTEM = """You have access to a search tool.
If you need to look something up, output: search[\"your query\"]
Otherwise output your final answer directly. Be concise."""

async def answer_with_tool(question: str) -> str:
    """One-shot tool dispatch: model may optionally search once, then answer."""
    messages = [
        {"role": "system", "content": WITH_TOOL_SYSTEM},
        {"role": "user",   "content": question},
    ]
    reply = await chat(messages)
    messages.append({"role": "assistant", "content": reply})

    match = re.search(r'search\[\"(.+?)\"\]', reply)  # <1>
    if match:
        observation = search_wikipedia(match.group(1))
        messages.append({
            "role": "user",
            "content": f"Observation: {observation}\n\nNow give the final answer concisely.",
        })
        reply = await chat(messages)  # <2>

    return reply

with_tool_answers = [await answer_with_tool(item["q"]) for item in QUESTIONS]

1. If the model chose to search, we execute the real Wikipedia lookup and inject the result as a new user turn.
2. With the observation in context, the model now answers from grounded evidence rather than parametric memory alone.

**Results.** Comparing both conditions against the known correct answers:

In [ ]:
#| code-fold: true
import textwrap
import pandas as pd

wrap = lambda s, w=35: "\n".join(textwrap.wrap(str(s), w))

rows = [
    {
        "Question":       wrap(item["q"]),
        "No Tool":        wrap(no_ans),
        "With Tool":      wrap(tool_ans),
        "Correct Answer": wrap(item["correct"]),
    }
    for item, no_ans, tool_ans in zip(QUESTIONS, no_tool_answers, with_tool_answers)
]

df = pd.DataFrame(rows)
with pd.option_context("display.max_colwidth", None, "display.width", 160):
    display(df)

Some answers are correct in both conditions &mdash; the knowledge is common enough that it is well-represented in parametric memory. The interesting cases are where the model *confidently* gives the wrong answer without tools but corrects itself once grounded. [The model is identical in both conditions]{.mark}: what changes is whether it has access to an external source of truth at inference time.

:::{.callout-caution}
Tool access shifts but does not eliminate hallucination. With poor retrieval (wrong article, truncated context, or a query that returns an unrelated page), the model can still confabulate a plausible synthesis of a bad observation. Context quality matters as much as context availability.

:::

---

&#9632;